## Week 6 Assignment

This guide will walk you through creating the Python notebook for Week 6's assignment on speaker diarization and identification. You will be building a pipeline that transcribes audio, extracts speaker embeddings, clusters these embeddings to identify potential speakers, and then attempts to label these speakers by comparing them to known speaker profiles.

**I. Setting up the Environment and Installing Libraries**

1.  **Start with a new Colab Notebook:** Open Google Colab and create a new Python 3 notebook. This will be your workspace for the assignment.

2.  **Install Required Libraries:**  You'll need to install several Python libraries to perform audio processing, speech transcription, speaker embedding extraction, and clustering. Use `pip install` commands within a code cell to install the following packages. Ensure you install the specific versions mentioned if provided in the original notebook description for compatibility. The key libraries are:
    *   `speechbrain`: A speech processing toolkit, specifically version `0.5.16`.
    *   `faster-whisper`: A faster implementation of the Whisper transcription model.
    *   `pyannote.audio`: A library for audio analysis, including speaker diarization.
    *   `whisper`: The original OpenAI Whisper model (may be needed as a dependency).
    *   `ctranslate2`: For efficient model translation and inference, version `4.4.0`.
    *   `librosa`: For audio analysis and feature extraction.
    *   `soundfile`: For reading and writing audio files.
    *   `scikit-learn` (`sklearn`): For machine learning tasks like KMeans clustering and cosine similarity calculations.

    *Remember to use `!pip install <library_name>` in a code cell to install each library. You can combine multiple installations in a single cell for convenience. The notebook may prompt you to restart the kernel. Please do so.*

    ```
    !pip install speechbrain==0.5.16 faster-whisper pyannote.audio whisper ctranslate2==4.4.0 librosa soundfile scikit-learn > /dev/null
    ```

**II. Importing Necessary Modules**

1.  **Import Python Libraries:** After successful installation, you need to import the modules and classes from these libraries that you'll be using in your code.

**III. Defining Helper Functions**

You will need to define several functions to structure your speaker diarization and identification pipeline. Each function will perform a specific task. Please refer to the Professor's notebook for guidance.

1.  **`convert_time(secs)` Function:**
    *   **Purpose:** This function is for converting time in seconds into a human-readable format, specifically `hh:mm:ss`.
    *   **Input:** It takes a floating-point number representing time in seconds (`secs`).
    *   **Output:** It should return a string representing the time in `hh:mm:ss` format. You will need to use the `datetime` module to achieve this conversion.

2.  **`transcribe_audio(audio_file, model_name="base", language="en", beam_size=5, best_of=5)` Function:**
    *   **Purpose:** This function handles the audio transcription process using the `faster-whisper` library.
    *   **Inputs:**
        *   `audio_file`: Path to the audio file you want to transcribe.
        *   `model_name`: Name of the Whisper model to use (e.g., "tiny", "base", "small"). Default is "base".
        *   `language`: Language of the audio for transcription (e.g., "en" for English). Default is "en".
        *   `beam_size`, `best_of`: Parameters to control the decoding process in Whisper for better transcription quality.
    *   **Process:**
        *   Load the specified Whisper model using `WhisperModel`. Please load the model with `compute_type="int8"` for efficiency.
        *   Use the model's `transcribe` method to transcribe the `audio_file`.  You'll need to set transcription options, including the `language`, `beam_size`, and `best_of` parameters.
        *   The `transcribe` method will return segments of transcribed audio.
        *   Format the output into a list of dictionaries. Each dictionary should represent a segment and contain:
            *   `start`: Start time of the segment in seconds.
            *   `end`: End time of the segment in seconds.
            *   `text`: (Optional for this assignment, but good to include) The transcribed text for the segment.
    *   **Output:** Return a list of segment dictionaries.

3.  **`extract_segment_embedding(audio_file, segment, total_duration, embedding_model)` Function:**
    *   **Purpose:** This function extracts a speaker embedding for a specific segment of an audio file using a pre-trained speaker embedding model.
    *   **Inputs:**
        *   `audio_file`: Path to the full audio file.
        *   `segment`: A dictionary representing an audio segment (with 'start' and 'end' times).
        *   `total_duration`: The total duration of the audio file in seconds.
        *   `embedding_model`: The pre-loaded speaker embedding model (from `speechbrain/spkrec-ecapa-voxceleb`).
    *   **Process:**
        *   Initialize the `Audio()` processor from `pyannote.audio`.
        *   Extract the `start` and `end` times from the `segment` dictionary. Ensure the `end` time does not exceed the `total_duration`.
        *   Create a `Segment` object from `pyannote.core` using the `start` and `end` times.
        *   Use the `audio.crop()` method to extract the waveform for the given segment from the `audio_file`. This method will return the waveform and the sample rate.
        *   The speaker embedding model expects input with a batch dimension. Add a batch dimension to the waveform using `waveform[None]`.
        *   Pass the waveform to the `embedding_model` to get the speaker embedding.
        *   Remove the batch dimension from the embedding using `squeeze()` to get a 1D embedding vector.
    *   **Output:** Return the speaker embedding as a NumPy array.  Handle potential errors within a `try-except` block and print traceback information if necessary.

4.  **`compute_segment_embeddings(audio_file, segments, embedding_model)` Function:**
    *   **Purpose:** This function iterates through all transcribed segments of an audio file and computes speaker embeddings for each segment.
    *   **Inputs:**
        *   `audio_file`: Path to the audio file.
        *   `segments`: List of segment dictionaries from the transcription step.
        *   `embedding_model`: The speaker embedding model.
    *   **Process:**
        *   Load the entire audio file using `librosa.load()` to get the total duration. Ensure you load it with a sample rate of 16000 Hz and as mono.
        *   Initialize an empty list to store embeddings.
        *   Loop through each `segment` in the `segments` list.
        *   For each segment, call the `extract_segment_embedding` function to get the embedding.
        *   Append the obtained embedding to the list.
        *   Finally, stack all the embeddings into a 2D NumPy array where each row is an embedding vector.
    *   **Output:** Return the 2D array of segment embeddings and the `duration` of the audio file.

5.  **`cluster_embeddings(embeddings, n_clusters)` Function:**
    *   **Purpose:** This function clusters the computed segment embeddings using KMeans to group segments likely spoken by the same speaker.
    *   **Inputs:**
        *   `embeddings`: 2D array of segment embeddings.
        *   `n_clusters`: The number of clusters to create (which corresponds to the assumed number of speakers).
    *   **Process:**
        *   Initialize a `KMeans` clustering object with the specified `n_clusters` and set `random_state=42` for reproducibility.
        *   Fit the KMeans model to the `embeddings`.
        *   Get the cluster labels assigned to each embedding using `clustering.labels_`.
        *   Optionally, get the cluster centroids using `clustering.cluster_centers_`.
    *   **Output:** Return the cluster `labels` and the `centroids`.

6.  **`compute_cluster_averages(embeddings, labels, n_clusters)` Function:**
    *   **Purpose:** Calculate the average embedding for each cluster. This average embedding represents a cluster's speaker profile.
    *   **Inputs:**
        *   `embeddings`: 2D array of segment embeddings.
        *   `labels`: Cluster labels assigned to each segment.
        *   `n_clusters`: The number of clusters.
    *   **Process:**
        *   Initialize an empty dictionary to store cluster average embeddings.
        *   Iterate through each cluster ID from `0` to `n_clusters - 1`.
        *   For each cluster ID, select all embeddings that belong to this cluster (based on `labels`).
        *   Calculate the mean of these embeddings along the embedding dimension to get the average embedding for the cluster.
        *   Store the average embedding in the dictionary with the cluster ID as the key.
    *   **Output:** Return a dictionary where keys are cluster IDs and values are the corresponding average embedding vectors.

7.  **`load_known_speaker_embeddings(known_speaker_files, embedding_model)` Function:**
    *   **Purpose:**  To load audio files of known speakers, compute their speaker embeddings, and store them for speaker identification.
    *   **Inputs:**
        *   `known_speaker_files`: A list of file paths to audio files of known speakers.
        *   `embedding_model`: The speaker embedding model.
    *   **Process:**
        *   Initialize an empty dictionary to store known speaker embeddings.
        *   Loop through each `file` in `known_speaker_files`.
        *   Extract a speaker label from the filename (e.g., if the file is "speaker_A.wav", the label is "speaker_A").
        *   Load the audio file using `librosa.load()` (at 16kHz, mono).
        *   Initialize `Audio()` processor. Create a `Segment` object covering the entire duration of the loaded audio. Crop the audio using this segment.
        *   Compute the speaker embedding for the whole audio file using the `embedding_model`.
        *   Store the computed embedding in the dictionary with the speaker label as the key.
    *   **Output:** Return a dictionary of known speaker embeddings where keys are speaker labels and values are their embedding vectors.

8.  **`assign_speaker_labels(cluster_avg_embeddings, known_speaker_embeddings, similarity_threshold=0.7)` Function:**
    *   **Purpose:**  Compare average embeddings of clusters with embeddings of known speakers to assign speaker labels to clusters.
    *   **Inputs:**
        *   `cluster_avg_embeddings`: Dictionary of average embeddings for each cluster.
        *   `known_speaker_embeddings`: Dictionary of embeddings for known speakers.
        *   `similarity_threshold`: Threshold for cosine similarity to consider a match reliable.
    *   **Process:**
        *   Initialize an empty dictionary to map cluster IDs to speaker labels.
        *   Iterate through each `cluster_id` and its `avg_emb` in `cluster_avg_embeddings`.
        *   For each cluster, calculate the cosine similarity between its `avg_emb` and each `known_emb` in `known_speaker_embeddings`. Use `cosine_similarity` from `sklearn.metrics.pairwise`. Remember to reshape embeddings to 2D arrays before using `cosine_similarity`.
        *   Find the known speaker with the highest cosine similarity to the current cluster.
        *   Check if the highest similarity score is greater than or equal to the `similarity_threshold`.
            *   If it is, assign the label of the best matching known speaker to the current `cluster_id` in the output dictionary.
            *   If it is not, assign the label "Unknown" to the current `cluster_id`.
    *   **Output:** Return a dictionary mapping cluster IDs to assigned speaker labels.

9.  **`run_pipeline(audio_file, embedding_model, known_speaker_embeddings, n_clusters=3, whisper_model_name="base", similarity_threshold=0.7)` Function:**
    *   **Purpose:** This function orchestrates the entire speaker diarization and identification pipeline, calling all the previously defined functions in the correct sequence.
    *   **Inputs:**
        *   `audio_file`: Path to the audio file to process.
        *   `embedding_model`: The pre-loaded speaker embedding model.
        *   `known_speaker_embeddings`: Dictionary of known speaker embeddings.
        *   `n_clusters`: Number of speakers to assume for clustering.
        *   `whisper_model_name`: Whisper model name for transcription.
        *   `similarity_threshold`: Similarity threshold for speaker identification.
    *   **Process:**
        *   Print a separator and a message indicating the processing of the current `audio_file`.
        *   Call `transcribe_audio` to get transcription segments.
        *   Call `compute_segment_embeddings` to get embeddings for each segment.
        *   Call `cluster_embeddings` to cluster the segment embeddings.
        *   Call `compute_cluster_averages` to get average embeddings per cluster.
        *   Call `assign_speaker_labels` to assign speaker labels to clusters.
        *   Annotate each segment with its cluster ID and assigned speaker ID.
        *   Format the results into a Pandas DataFrame. The DataFrame should contain columns for "Start", "End", "Cluster", and "Speaker_ID". Group consecutive segments with the same speaker ID to produce a diarization-like output. Convert start and end times to `hh:mm:ss` format using `convert_time`.
        *   Print the resulting DataFrame.
    *   **Output:** Return the results DataFrame, the number of segments transcribed, and the cluster centroids.

**IV. Loading Models and Data**

1.  **Set Device:** Please use the CPU to run your code as GPUs are not required for this module. Errors caused by GPU runtimes will not be considered.

2.  **Load Speaker Embedding Model:** Load the pre-trained speaker embedding model "speechbrain/spkrec-ecapa-voxceleb" using `PretrainedSpeakerEmbedding`. Move the model to the device you set in the previous step.

3.  **Load Known Speaker Audio Files and Compute Embeddings:**
    *   Create a list of filenames for the known speaker audio files (e.g., "speaker_A.wav", "speaker_B.wav", etc.). You will need to ensure the audio files are available in your Colab environment.  You can download all necessary audio files, including `sample.wav`, `sample_noisy.wav`, `speaker_A.wav`, `speaker_B.wav`, `speaker_C.wav`, `speaker_D.wav`, and `speaker_E.wav`, as a single archive file named `wavs.tar.gz` from this link: [https://drive.google.com/file/d/1ZcSTNF43seMPo0nF8sdXY5922Luc2bqn/view?usp=sharing]. Once downloaded, use the command `!tar -xvf wavs.tar.gz` in a code cell within your Colab notebook to extract these audio files into your working directory.
    *   Call `load_known_speaker_embeddings` function, passing the list of filenames and the embedding model. This will give you a dictionary of known speaker embeddings.

**V. Processing Audio Samples**

1.  **Process "clean" audio sample (`sample.wav`):**
    *   Set the `clean_audio` variable to the filename "sample.wav".
    *   Call the `run_pipeline` function with `clean_audio`, the embedding model, known speaker embeddings, and appropriate parameters like `n_clusters=3`, `whisper_model_name="base"`, and `similarity_threshold=0.7`.
    *   Store the returned DataFrame, number of segments, and cluster centroids in variables like `df_clean`, `n_segments_clean`, and `centroids_clean`.

2.  **Process "noisy" audio sample (`sample_noisy.wav`):**
    *   Set the `noisy_audio` variable to the filename "sample_noisy.wav".
    *   Call `run_pipeline` again, this time with `noisy_audio` and the same models and parameters as before.
    *   Store the results in variables like `df_noisy`, `n_segments_noisy`, and `centroids_noisy`.


After running the pipeline for both audio samples, you will be ready to answer the questions.

In [ ]:
!pip install speechbrain==0.5.16 faster-whisper pyannote.audio whisper ctranslate2==4.4.0 librosa soundfile scikit-learn > /dev/null

In [ ]:
import os
import time
import datetime
import traceback
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
from faster_whisper import WhisperModel
from pyannote.audio import Audio
from pyannote.audio.pipelines.speaker_verification import PretrainedSpeakerEmbedding
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!tar -xvf wavs.tar.gz

speaker_A.wav
speaker_B.wav
speaker_C.wav
speaker_D.wav
speaker_E.wav
sample.wav
sample_noisy.wav


In [ ]:
from pyannote.core import Segment  # Make sure to import Segment

def convert_time(secs):
    """Convert seconds to hh:mm:ss format."""
    return str(datetime.timedelta(seconds=round(secs)))

def transcribe_audio(audio_file, model_name="base", language="en", beam_size=5, best_of=5):
    """
    Transcribe audio using fast_whisper.
    Returns a list of segments; each segment is a dict with 'start', 'end', and (optional) 'text'.
    """
    print(f"Loading Whisper model '{model_name}' with int8 quantization ...")
    model = WhisperModel(model_name, compute_type="int8")
    print("Transcribing:", audio_file)
    options = dict(language=language, beam_size=beam_size, best_of=best_of)
    transcribe_options = dict(task="transcribe", **options)

    segments_raw, info = model.transcribe(audio_file, **transcribe_options)

    segments = []
    for seg in segments_raw:
        segments.append({
            "start": seg.start,
            "end": seg.end,
            "text": seg.text  # if needed
        })
    print(f"Number of segments: {len(segments)}")
    return segments

def extract_segment_embedding(audio_file, segment, total_duration, embedding_model):
    """
    Crop the audio file to the segment and compute its speaker embedding.
    """
    try:
        audio = Audio()
        start = segment["start"]
        # Ensure we do not exceed duration
        end = min(total_duration, segment["end"])
        # Create a Segment object instead of a tuple.
        clip = Segment(start, end)
        # Crop returns waveform and sample rate.
        waveform, sr = audio.crop(audio_file, clip)
        # The embedding model expects a batch dimension (i.e., an extra axis)
        emb = embedding_model(waveform[None])
        return emb.squeeze()  # shape (192,)
    except Exception as e:
        traceback.print_exc()
        raise RuntimeError("Error during segment embedding") from e

def compute_segment_embeddings(audio_file, segments, embedding_model):
    """
    Compute embeddings for all segments.
    """
    # Load full audio to get duration
    waveform, sr = librosa.load(audio_file, sr=16000, mono=True)
    duration = len(waveform) / sr
    embeddings = []
    for seg in segments:
        emb = extract_segment_embedding(audio_file, seg, duration, embedding_model)
        embeddings.append(emb)
    embeddings = np.vstack(embeddings)  # shape: (n_segments, embedding_dim)
    return embeddings, duration

def cluster_embeddings(embeddings, n_clusters):
    """
    Cluster the segment embeddings using KMeans.
    """
    clustering = KMeans(n_clusters=n_clusters, random_state=42)
    clustering.fit(embeddings)
    labels = clustering.labels_
    centroids = clustering.cluster_centers_
    return labels, centroids

def compute_cluster_averages(embeddings, labels, n_clusters):
    """
    Compute the average embedding for each cluster.
    """
    cluster_avg = {}
    for k in range(n_clusters):
        cluster_embeddings = embeddings[labels == k]
        cluster_avg[k] = np.mean(cluster_embeddings, axis=0)
    return cluster_avg


def load_known_speaker_embeddings(known_speaker_files, embedding_model):
    """
    For each known speaker audio file, compute and store the embedding.
    The speaker label is derived from the file name (e.g., 'speaker_A').
    Returns a dictionary mapping speaker label to embedding vector.
    """
    known_embeddings = {}
    for file in known_speaker_files:
        # Derive speaker label from filename
        label = os.path.splitext(os.path.basename(file))[0]
        print(f"Processing known speaker: {label}")
        waveform, sr = librosa.load(file, sr=16000, mono=True)
        # Use entire file as a single segment.
        audio = Audio()
        # Create a Segment object instead of a tuple
        clip = Segment(0, len(waveform) / sr)
        # Crop returns waveform with shape (n_samples,) and sample rate.
        segment_waveform, _ = audio.crop(file, clip)
        emb = embedding_model(segment_waveform[None])
        known_embeddings[label] = emb.squeeze()
    return known_embeddings


def assign_speaker_labels(cluster_avg_embeddings, known_speaker_embeddings, similarity_threshold=0.7):
    """
    Compare each cluster's average embedding with the known speaker embeddings.
    Assign the known speaker label if the maximum cosine similarity exceeds the threshold;
    otherwise, mark as 'Unknown'.
    Returns a dictionary mapping cluster id to identified speaker label.
    """
    cluster_to_speaker = {}
    for cluster_id, avg_emb in cluster_avg_embeddings.items():
        similarities = {}
        for speaker_label, known_emb in known_speaker_embeddings.items():
            # Compute cosine similarity; reshape to 2D arrays as required
            sim = cosine_similarity(avg_emb.reshape(1, -1), known_emb.reshape(1, -1))[0][0]
            similarities[speaker_label] = sim
        # Find the best match
        best_speaker = max(similarities, key=similarities.get)
        best_sim = similarities[best_speaker]
        if best_sim >= similarity_threshold:
            cluster_to_speaker[cluster_id] = best_speaker
        else:
            cluster_to_speaker[cluster_id] = "Unknown"
        print(f"Cluster {cluster_id}: best match {best_speaker} with similarity {best_sim:.3f}")
    return cluster_to_speaker

In [ ]:
def run_pipeline(audio_file, embedding_model, known_speaker_embeddings, n_clusters=3,
                 whisper_model_name="base", similarity_threshold=0.7):
    """
    Run the complete pipeline:
      - Transcription via fast_whisper.
      - Segment embedding extraction.
      - Clustering.
      - Speaker identification.
    Returns a DataFrame with segment timings and assigned speaker labels.
    """
    print("\n" + "="*60)
    print(f"Processing file: {audio_file}")

    # Step 1: Transcription
    segments = transcribe_audio(audio_file, model_name=whisper_model_name)

    # Step 2: Compute embeddings for each segment
    embeddings, duration = compute_segment_embeddings(audio_file, segments, embedding_model)

    # Step 3: Cluster the segment embeddings
    labels, centroids = cluster_embeddings(embeddings, n_clusters)
    cluster_avg_embeddings = compute_cluster_averages(embeddings, labels, n_clusters)

    # Step 4: Identify speakers by comparing cluster average with known speaker embeddings
    cluster_to_speaker = assign_speaker_labels(cluster_avg_embeddings, known_speaker_embeddings, similarity_threshold)

    # Annotate segments with cluster label and identified speaker label
    for i, seg in enumerate(segments):
        seg["cluster"] = int(labels[i])
        seg["speaker_id"] = cluster_to_speaker[labels[i]]

    # Create a DataFrame for results
    results = {"Start": [], "End": [], "Cluster": [], "Speaker_ID": []}
    # Group segments by changes in assigned speaker to simulate diarization output
    prev_speaker = None
    for seg in segments:
        current = seg["speaker_id"]
        if prev_speaker is None or current != prev_speaker:
            results["Start"].append(convert_time(seg["start"]))
            results["Cluster"].append(seg["cluster"])
            results["Speaker_ID"].append(current)
            if prev_speaker is not None:
                # Use previous segment's end as the end time for the previous speaker.
                results["End"].append(convert_time(prev_end))
        prev_speaker = current
        prev_end = seg["end"]
    # Append the end time for the last segment.
    results["End"].append(convert_time(prev_end))

    df_results = pd.DataFrame(results)
    print("Diarization and Speaker Identification Results:")
    print(df_results)
    return df_results, len(segments), centroids

In [ ]:
# Set up device and load the speaker embedding model from Hugging Face via pyannote
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model = PretrainedSpeakerEmbedding("speechbrain/spkrec-ecapa-voxceleb", device=device)

/usr/local/lib/python3.11/dist-packages/speechbrain/utils/checkpoints.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device), strict=Fal

In [ ]:
# Load known speaker embeddings
known_speaker_files = ["speaker_A.wav", "speaker_B.wav", "speaker_C.wav", "speaker_D.wav", "speaker_E.wav"]
known_speaker_embeddings = load_known_speaker_embeddings(known_speaker_files, embedding_model)

Processing known speaker: speaker_A
Processing known speaker: speaker_B
Processing known speaker: speaker_C
Processing known speaker: speaker_D
Processing known speaker: speaker_E


In [ ]:
# Process the clean sample
clean_audio = "sample.wav"
df_clean, n_segments_clean, centroids_clean = run_pipeline(
    clean_audio, embedding_model, known_speaker_embeddings, n_clusters=3,
    whisper_model_name="base", similarity_threshold=0.7)

# Process the noisy sample
noisy_audio = "sample_noisy.wav"
df_noisy, n_segments_noisy, centroids_noisy = run_pipeline(
    noisy_audio, embedding_model, known_speaker_embeddings, n_clusters=3,
    whisper_model_name="base", similarity_threshold=0.7)


Processing file: sample.wav
Loading Whisper model 'base' with int8 quantization ...
Transcribing: sample.wav
Number of segments: 6
Cluster 0: best match speaker_A with similarity 0.689
Cluster 1: best match speaker_C with similarity 0.797
Cluster 2: best match speaker_B with similarity 0.858
Diarization and Speaker Identification Results:
     Start      End  Cluster Speaker_ID
0  0:00:00  0:00:06        2  speaker_B
1  0:00:08  0:00:14        1  speaker_C
2  0:00:15  0:00:17        0  speaker_A
3  0:00:18  0:00:24        1  speaker_C
4  0:00:25  0:00:29        0  speaker_A
5  0:00:31  0:00:36        2  speaker_B

Processing file: sample_noisy.wav
Loading Whisper model 'base' with int8 quantization ...
Transcribing: sample_noisy.wav
Number of segments: 8
Cluster 0: best match speaker_A with similarity 0.581
Cluster 1: best match speaker_C with similarity 0.545
Cluster 2: best match speaker_B with similarity 0.676
Diarization and Speaker Identification Results:
     Start      End  Clu

## Questions

### Q1
**For the clean audio sample ("sample.wav"), the three clusters have best cosine similarity scores of a, b, and c. Calculate the average cosine similarity score across all clusters (round to three decimal places).**

- **Answer:** `[0.7, 0.8]`


In [ ]:
# Calculate average similarity for clean sample
clean_similarities = [0.689, 0.797, 0.858]
avg_clean_similarity = round(sum(clean_similarities) / len(clean_similarities), 3)
print(f"Average cosine similarity for clean sample clusters: {avg_clean_similarity}")

Average cosine similarity for clean sample clusters: 0.781


### Q2
**For the noisy audio sample ("sample_noisy.wav"), the clusters have best similarity scores of x, y, z. Calculate the average cosine similarity score across these clusters (round to three decimal places).**

- **Answer:** `[0.55, 0.65]`

In [ ]:
# Calculate average similarity for noisy sample
noisy_similarities = [0.581, 0.545, 0.676]
avg_noisy_similarity = round(sum(noisy_similarities) / len(noisy_similarities), 3)
print(f"Average cosine similarity for noisy sample clusters: {avg_noisy_similarity}")

Average cosine similarity for noisy sample clusters: 0.601

Processing file: sample_noisy.wav
Loading Whisper model 'base' with int8 quantization ...
Transcribing: sample_noisy.wav
Number of segments: 8
Cluster 0: best match speaker_A with similarity 0.581
Cluster 1: best match speaker_C with similarity 0.545
Cluster 2: best match speaker_B with similarity 0.676
Diarization and Speaker Identification Results:
     Start      End  Cluster Speaker_ID
0  0:00:00  0:00:05        2  speaker_B
1  0:00:08  0:00:29        1    Unknown
2  0:00:30  0:00:35        2  speaker_B


### Q3
**If a similarity threshold of 0.7 is required for reliable speaker identification, how many clusters in the clean sample meet or exceed this threshold?**

- Options:
  - (A) 0
  - (B) 1
  - (C) 2
  - (D) 3

- **Answer:** `2`


In [ ]:
threshold = 0.7
clean_above_threshold = sum(1 for sim in clean_similarities if sim >= threshold)
print(f"Number of clean clusters above threshold ({threshold}): {clean_above_threshold}")

Number of clean clusters above threshold (0.7): 2


### Q4
**Using the same similarity threshold (0.7), how many clusters in the noisy sample would be considered reliably matched?**

- Options:
  - (A) 0
  - (B) 1
  - (C) 2
  - (D) 3

- **Answer:** `0`

In [ ]:
noisy_above_threshold = sum(1 for sim in noisy_similarities if sim >= threshold)
print(f"Number of noisy clusters above threshold ({threshold}): {noisy_above_threshold}")

Number of noisy clusters above threshold (0.7): 0


### Q5
**The transcription produced m segments for "sample.wav" and n segments for "sample_noisy.wav." Calculate the percentage increase in the number of segments in the noisy sample relative to the clean sample (round to two decimal place).**

- **Answer:** `[0.3, 0.35]` or `[30, 35]`

In [ ]:
# Calculate percentage increase in segments
percentage_increase = round(((n_segments_noisy - n_segments_clean) / n_segments_clean), 2)
print(f"Percentage increase in segments for noisy sample: {percentage_increase}")

Percentage increase in segments for noisy sample: 0.33


### Q6
**Using the diarization results for the clean sample, compute the total duration (in seconds) covered by all the transcribed segments.**

- **Answer:** `[27, 31]`

In [ ]:
segment_durations = [6, 6, 2, 6, 4, 5]
total_duration = sum(segment_durations)
print(f"Total duration of transcribed segments (clean sample): {total_duration} seconds")

Total duration of transcribed segments (clean sample): 29 seconds


### Q7
**If the similarity threshold were reduced from 0.7 to 0.65, which cluster in the noisy sample would now be assigned a known speaker label instead of "Unknown"?**

- Options:
  - (A) Cluster 0
  - (B) Cluster 1
  - (C) Cluster 2
  - (D) None of the clusters

- **Answer:** `Cluster 2`

In [ ]:
reduced_threshold = 0.65
noisy_cluster_2_similarity = 0.676 # From question description
cluster_2_above_reduced_threshold = noisy_cluster_2_similarity > reduced_threshold
print(f"Cluster 2 similarity ({noisy_cluster_2_similarity}) above reduced threshold ({reduced_threshold}): {cluster_2_above_reduced_threshold}")

Cluster 2 similarity (0.676) above reduced threshold (0.65): True


### Q8
**Examining the clustering results for the clean sample, which cluster shows the highest confidence (i.e. the highest cosine similarity), and what known speaker label is it matched with?**

- Options:
  - (A) Cluster 0 → speaker_A
  - (B) Cluster 1 → speaker_C
  - (C) Cluster 2 → speaker_B
  - (D) Cluster 2 → Unknown

- **Answer:** `Cluster 2 → speaker_B`


In [ ]:
# From question description, Cluster 2 has highest similarity 0.858 and label should be speaker_B based on typical output
highest_similarity_cluster = "Cluster 2"
matched_speaker_label = "speaker_B"
highest_similarity_value = 0.858

print(f"Highest similarity cluster: {highest_similarity_cluster} with similarity {highest_similarity_value} and label {matched_speaker_label}")

Highest similarity cluster: Cluster 2 with similarity 0.858 and label speaker_B
